# Training GPT-2 from scratch

This notebook trains a GPT-2-architecture language model from random initialisation on
TinyShakespeare and watches it go from noise to English.

Everything here runs on **Colab's free T4** (or on CPU, more slowly). Nothing costs money
and no API key is needed.

**Runtime → Change runtime type → T4 GPU** before you start.

In [ ]:
# Colab setup. Skip the clone if you are running this locally from the repo.
import os, sys

if not os.path.exists("gpt2-from-scratch"):
    !git clone -q https://github.com/bharathbagadhi/gpt2-from-scratch.git
%cd gpt2-from-scratch
!pip install -q -e . 2>/dev/null

sys.path.insert(0, "src")

import torch
print("torch", torch.__version__)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 1. The data

TinyShakespeare is 1.1 MB of the complete works, concatenated. Small enough to download in a
second, large enough that a small model learns real structure from it.

We tokenise it two ways:
- **char** — one token per character, 65-symbol vocabulary. Converges fast; ideal for
  watching the training loop work.
- **BPE** — GPT-2's real 50,257-token byte-level BPE. What you need for anything transferable.

In [ ]:
!python scripts/prepare_data.py --dataset tinyshakespeare --tokenizer char
!python scripts/prepare_data.py --dataset tinyshakespeare

### What a tokenizer actually does

Byte-level BPE compresses English to roughly 4 characters per token. Because it operates on
*bytes*, every possible input is encodable — there is no `<unk>` token and no text it cannot
represent.

In [ ]:
from gpt2.tokenizer import BPETokenizer, CharTokenizer

bpe = BPETokenizer()
text = "To be, or not to be, that is the question."

ids = bpe.encode(text)
print(f"vocab size : {bpe.vocab_size:,}")
print(f"text       : {text}")
print(f"tokens     : {ids}")
print(f"pieces     : {[bpe.decode([i]) for i in ids]}")
print(f"compression: {len(text) / len(ids):.2f} chars/token")
print(f"roundtrip  : {bpe.decode(ids) == text}")

### Input and target are the same stream, shifted by one

This is the whole of the language-modelling objective. For a window of `B*T+1` tokens, `x` is
the first `B*T` and `y` is the same window shifted left by one, so `y[i]` is the token that
follows `x[i]`. One forward pass therefore supplies `B*T` supervised examples, not `B` — this
is why language models are so sample-efficient per unit of compute.

In [ ]:
from gpt2.data import TokenLoader

char_tok = CharTokenizer.load("data/tinyshakespeare_char/vocab.json")
loader = TokenLoader("data/tinyshakespeare_char/train.bin", batch_size=2, block_size=16)
x, y = loader.next_batch()

print("x[0]:", repr(char_tok.decode(x[0].tolist())))
print("y[0]:", repr(char_tok.decode(y[0].tolist())))
print()
for i in range(6):
    ctx = char_tok.decode(x[0, :i+1].tolist())
    tgt = char_tok.decode([y[0, i].item()])
    print(f"  given {ctx!r:<20} predict {tgt!r}")

## 2. The model

`GPTConfig` holds the architecture. The defaults are GPT-2 small exactly; here we build a
smaller one so it trains in minutes rather than hours.

In [ ]:
from gpt2 import GPT, GPTConfig

full = GPT(GPTConfig())
print(f"GPT-2 small : {full.num_params():,} parameters")
print()

# Where they live.
groups = {
    "token embedding (tied with lm_head)": full.transformer.wte.weight.numel(),
    "position embedding": full.transformer.wpe.weight.numel(),
    "attention": sum(p.numel() for n, p in full.named_parameters() if ".attn." in n),
    "mlp": sum(p.numel() for n, p in full.named_parameters() if ".mlp." in n),
}
for name, n in groups.items():
    print(f"  {name:<38} {n:>12,}  ({n / full.num_params():5.1%})")

del full

### Weight tying, and why the loss starts at ln(V)

`wte` (token → vector) and `lm_head` (vector → logits) are the *same* matrix. That saves 38.6M
parameters and improves perplexity, because the two directions regularise each other.

At initialisation the model knows nothing, so it should assign roughly uniform probability to
every token — cross-entropy `≈ ln(vocab_size)`. If your loss starts anywhere else, the
initialisation is broken, and you have found out in one second instead of after an hour of
training.

In [ ]:
import math

cfg = GPTConfig(block_size=128, vocab_size=char_tok.vocab_size,
                n_layer=6, n_head=6, n_embd=192)
model = GPT(cfg)

print("wte is lm_head:", model.transformer.wte.weight is model.lm_head.weight)
print(f"parameters    : {model.num_params():,}")

x, y = TokenLoader("data/tinyshakespeare_char/train.bin", 4, 128).next_batch()
_, loss = model(x, y)
print(f"\ninitial loss  : {loss.item():.4f}")
print(f"ln(vocab_size): {math.log(cfg.vocab_size):.4f}   <- they should match")

### What it generates before training

Uniform noise over the character vocabulary. Worth looking at once so the "after" is meaningful.

In [ ]:
import torch

prompt = torch.tensor([char_tok.encode("ROMEO:")], dtype=torch.long)
print(char_tok.decode(model.generate(prompt, 200, temperature=1.0)[0].tolist()))

## 3. Train it

On a T4 this is about 2 minutes. On CPU, roughly 12.

Watch the loss: it drops fast at first (the model learns character *frequency* — that `e` and
space are common), then more slowly as it learns spelling, then word boundaries, then the
speaker-name-colon-newline structure of a play.

In [ ]:
!python scripts/train.py \
    --preset gpt2-nano --n_layer 6 --n_head 6 --n_embd 192 \
    --data_dir data/tinyshakespeare_char --vocab_size 65 \
    --batch_size 32 --block_size 128 \
    --max_steps 2500 --warmup_steps 100 --learning_rate 2e-3 \
    --eval_interval 250 --eval_iters 25 --log_interval 100 \
    --out_dir out/demo

## 4. What it generates now

In [ ]:
from gpt2.pretrained import load_checkpoint

trained = load_checkpoint("out/demo/ckpt_final.pt", device="cpu")

for temp in (0.5, 0.8, 1.2):
    out = trained.generate(prompt, 300, temperature=temp, top_k=40)
    print(f"\n{'=' * 70}\ntemperature = {temp}\n{'=' * 70}")
    print(char_tok.decode(out[0].tolist()))

**Temperature** rescales the logits before the softmax. Below 1 it sharpens the distribution
(more repetitive, more confident); above 1 it flattens it (more varied, more mistakes). At 0
it is pure argmax and the model loops almost immediately.

## 5. Measure it

Perplexity is `exp(cross_entropy)` — the effective number of characters the model is choosing
between at each position. A uniform model over this 65-symbol vocabulary would score 65.

The gap between train and validation loss is the honest read on overfitting. On 1 MB of text
a larger model would show a much wider gap.

In [ ]:
from gpt2.data import build_loaders

train_loader, val_loader = build_loaders("data/tinyshakespeare_char", 16, 128)

@torch.no_grad()
def mean_loss(m, loader, iters=25):
    m.eval()
    return sum(m(*loader.next_batch())[1].item() for _ in range(iters)) / iters

tr = mean_loss(trained, train_loader)
va = mean_loss(trained, val_loader)
print(f"train loss : {tr:.4f}  (perplexity {math.exp(tr):5.2f})")
print(f"val   loss : {va:.4f}  (perplexity {math.exp(va):5.2f})")
print(f"uniform    : {math.log(65):.4f}  (perplexity {65:5.2f})   <- where we started")
print(f"\ntrain/val gap: {va - tr:.4f}")

### The training curve

Re-run the sweep across the checkpoints the training loop saved, or just plot the losses
printed above against the run's log. Here we plot the eval history recorded during training.

In [ ]:
import matplotlib.pyplot as plt

# The Trainer returns its eval history; re-run in-process to capture it.
from gpt2.config import GPTConfig, TrainConfig
from gpt2.train import Trainer

mcfg = GPTConfig(block_size=128, vocab_size=65, n_layer=4, n_head=4, n_embd=128)
tcfg = TrainConfig(
    data_dir="data/tinyshakespeare_char", out_dir="out/curve",
    batch_size=32, block_size=128, max_steps=600, warmup_steps=50,
    learning_rate=3e-3, eval_interval=50, eval_iters=20, log_interval=200,
)
hist = Trainer(mcfg, tcfg).train()["history"]

steps = [h["step"] for h in hist]
plt.figure(figsize=(7, 4))
plt.plot(steps, [h["train"] for h in hist], label="train", marker="o", ms=3)
plt.plot(steps, [h["val"] for h in hist], label="val", marker="s", ms=3)
plt.axhline(math.log(65), ls="--", c="gray", lw=1, label="uniform, ln(65)")
plt.xlabel("step"); plt.ylabel("cross-entropy loss")
plt.title("Training a 4-layer GPT on TinyShakespeare")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 6. Scaling up

The same script trains the full 124M model. On a free T4:

```bash
python scripts/train.py --preset gpt2 \
    --data_dir data/tinyshakespeare \
    --batch_size 4 --block_size 512 --total_batch_size 65536 \
    --max_steps 3000 --learning_rate 6e-4
```

`--total_batch_size` is in **tokens**; gradient accumulation is derived from it, so the
optimisation is identical on one T4 or eight A100s — only wall-clock time differs.

With more than one GPU:

```bash
torchrun --standalone --nproc_per_node=8 scripts/train.py --strategy ddp
```

Next: `02_pretrained_and_finetune.ipynb`, which loads OpenAI's actual GPT-2 weights into this
implementation.